# Review 2: clean replay exposure and held-out novelty

Import this notebook into Kaggle, enable **Internet** and **GPU (T4)**, and attach the private `cic-collection.parquet`. Run top to bottom. Part A compares unchanged shuffled Task 2 + replay training with optional inverse-frequency class-balanced draws from the **same 500 stored replay rows**. Part B evaluates maximum-confidence novelty using the five-class Task 1 model and genuinely held-out Task 2 classes. The combined flow collection has unverified per-row source provenance. This is a single-seed balanced-subset study, not deployment/full-data training. No execution outputs are stored here.

Strategy selection is predeclared: higher **clean seen-class validation macro-F1**, then lower validation Benign false-positive rate for an exact tie. Test results and novelty outcomes never choose the arm or its settings. The fifth-percentile novelty threshold is predeclared independently.


## 1. Clone and pin the published source

The full SHA contains the backward-compatible sampler, class-incremental task/replay code, confidence calibration, and novelty metrics. `main` does not contain these APIs.


In [ ]:
from pathlib import Path
import subprocess, sys
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = '4e46f7f8ded77612210237e8dc9f6ea8e83730da'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-balance-novelty')
if REPO_DIR.exists():
    raise RuntimeError('Repository folder already exists; inspect it or use a fresh session.')
subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
print('Verified source SHA:', checked_out_sha)


## 2. Inspect the runtime and test the sampler before scanning data

Keep Kaggle's CUDA-enabled PyTorch. Record actual versions; do not install the local requirements file over it. Run the synthetic sampler unit suite here: it exercises default shuffled compatibility, seeded weighted draws, epoch-to-epoch generator advancement, and a tiny CPU fit. These checks are functional, not NIDS results.


In [ ]:
from dataclasses import asdict, replace
from datetime import datetime, timezone
from importlib.metadata import version
from time import perf_counter
import hashlib, json, platform
import numpy as np
import pandas as pd
import pyarrow as pa
import matplotlib.pyplot as plt
import torch
from src.data import BROAD_CLASS_ORDER, sample_class_balanced_indices, stratified_split_indices, materialize_selected_rows
from src.continual_learning import TASK1_CLASS_IDS, TASK2_CLASS_IDS, prepare_two_task_dataset, select_balanced_replay
from src.models import ModelConfig, TabularTransformerClassifier
from src.evaluation import classification_metrics, forgetting_metrics, novelty_metrics
from src.novelty import calibrate_confidence_threshold, detect_unknown
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required; enable Kaggle GPU and restart.')
assert pa.array([1, 2]).to_pylist() == [1, 2] and pd.Series([1, 2]).sum() == 3
assert torch.tensor([1.0], device='cuda').is_cuda
environment = {'python': platform.python_version(), 'packages': {name: version(name) for name in ('numpy', 'pandas', 'pyarrow', 'torch', 'matplotlib')}, 'torch_cuda_build': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0)}
print('Environment:', environment)
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_training_sampling', 'tests.test_novelty_evaluation', '-v'], cwd=REPO_DIR, check=True)
print('Synthetic sampler and novelty API checks passed in this runtime')


## 3. Locate and checksum the attached collection

Edit `PARQUET_PATH` if there is zero or more than one candidate. Requiring the reviewed checksum avoids silently changing the experiment input. Hashing scans the full file.


In [ ]:
PARQUET_PATH = None  # e.g. '/kaggle/input/private-collection/cic-collection.parquet'
EXPECTED_INPUT_SHA256 = '666af53c788c79312b421c607289cb555c2335ae51ba39b2700a357595fa8dba'
candidates = sorted(Path('/kaggle/input').rglob('cic-collection.parquet'))
print('Candidates:', [str(path) for path in candidates])
if PARQUET_PATH is None:
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one attached Parquet file; found {len(candidates)}. Set PARQUET_PATH.')
    parquet_path = candidates[0]
else:
    parquet_path = Path(PARQUET_PATH)
    if not parquet_path.is_file() or not parquet_path.is_relative_to(Path('/kaggle/input')):
        raise ValueError('PARQUET_PATH must be an attached file under /kaggle/input')
start = perf_counter(); digest = hashlib.sha256()
with parquet_path.open('rb') as source_file:
    for chunk in iter(lambda: source_file.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
input_sha256 = digest.hexdigest(); checksum_seconds = perf_counter() - start
print('Input bytes:', parquet_path.stat().st_size, 'SHA-256:', input_sha256)
if input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError('Attached input differs from the reviewed combined collection.')


## 4. Build the same frozen Task 1→Task 2 stream and 500-row memory

The seeded 2,000-per-broad-class sampler still scans the Parquet labels. The existing 70/15/15 split uses seed 42. `prepare_two_task_dataset` fits imputation/scaling **only on Task 1 training rows**, then freezes it for all Task 2/validation/test rows. Global IDs stay 0–4 then 5–7. Select 100 unique Task 1 training rows per old class, excluding validation/test raw row IDs. Balanced sampling later changes *draw exposure*, not the stored 500-row memory.


In [ ]:
start = perf_counter(); sample = sample_class_balanced_indices(parquet_path, per_class_limit=2000, seed=42); sampling_seconds = perf_counter() - start
assert sample.class_order == BROAD_CLASS_ORDER
split = stratified_split_indices(sample.labels, seed=42)
start = perf_counter(); selected = materialize_selected_rows(parquet_path, sample.row_indices)
assert np.array_equal(selected.labels, sample.labels) and np.array_equal(selected.row_indices, sample.row_indices)
tasks = prepare_two_task_dataset(selected, split); preparation_seconds = perf_counter() - start
assert tasks.task1.class_ids == TASK1_CLASS_IDS and tasks.task2.class_ids == TASK2_CLASS_IDS
assert tasks.class_names == dict(enumerate(BROAD_CLASS_ORDER))
validation_rows = tasks.seen_validation.row_indices; test_rows = tasks.combined_test.row_indices
replay = select_balanced_replay(tasks.task1.train, class_ids=TASK1_CLASS_IDS, per_class=100, seed=42, forbidden_row_indices=np.concatenate((validation_rows, test_rows)))
assert replay.features.shape == (500, 54) and len(np.unique(replay.row_indices)) == 500
assert not np.intersect1d(replay.row_indices, validation_rows).size and not np.intersect1d(replay.row_indices, test_rows).size
for name in ('task1', 'task2'):
    task = getattr(tasks, name)
    for part_name in ('train', 'validation', 'test'):
        part = getattr(task, part_name)
        assert part.features.dtype == np.float32 and np.isfinite(part.features).all()
        assert set(np.unique(part.labels)) == set(task.class_ids)
        print(name, part_name, part.features.shape, 'global IDs:', np.unique(part.labels).tolist())
print('Class mapping:', tasks.class_names, 'replay counts:', replay.class_counts)


## 5. Create the timestamped reproducibility record

Store source SHA, environment, input checksum, original selected/split row IDs, Task-1-only preprocessing state, and the unchanged 500-row replay buffer. Generated artifacts stay under `/kaggle/working`, not Git.


In [ ]:
OUTPUT_DIR = Path('/kaggle/working') / ('review2_balance_novelty_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
np.savez_compressed(OUTPUT_DIR / 'row_indices.npz', selected_original=sample.row_indices, train_positions=split.train, validation_positions=split.validation, test_positions=split.test, task1_train_original=tasks.task1.train.row_indices, task2_train_original=tasks.task2.train.row_indices, validation_original=validation_rows, test_original=test_rows)
np.savez_compressed(OUTPUT_DIR / 'replay_buffer.npz', features=replay.features, labels=replay.labels, original_rows=replay.row_indices, task1_train_positions=replay.source_train_positions)
preprocessing = {'fit_scope': 'Task 1 training rows only', 'feature_names': list(tasks.preprocessor.feature_names), 'imputation_values': tasks.preprocessor.imputation_values.tolist(), 'means': tasks.preprocessor.means.tolist(), 'scales': tasks.preprocessor.scales.tolist()}
(OUTPUT_DIR / 'preprocessing.json').write_text(json.dumps(preprocessing, indent=2), encoding='utf-8')
manifest = {'purpose': 'clean replay exposure comparison and held-out maximum-confidence novelty evaluation', 'source_sha': checked_out_sha, 'environment': environment, 'input': {'path': str(parquet_path), 'bytes': parquet_path.stat().st_size, 'sha256': input_sha256, 'provenance': 'combined flow collection; per-row source provenance unverified'}, 'sampling': {'per_class_limit': 2000, 'seed': 42, 'available_counts': sample.available_counts}, 'split': {'seed': 42, 'fractions': [0.70, 0.15, 0.15]}, 'class_names': {str(key): value for key, value in tasks.class_names.items()}, 'task1_class_ids': list(TASK1_CLASS_IDS), 'task2_class_ids': list(TASK2_CLASS_IDS), 'preprocessing_fit_scope': 'Task 1 training only', 'replay': {'seed': 42, 'stored_unique_rows': len(replay.row_indices), 'counts': replay.class_counts}, 'timings_seconds': {'checksum': checksum_seconds, 'sampling': sampling_seconds, 'materialization_and_preparation': preparation_seconds}, 'strategy_selection_predeclared': {'primary': 'higher clean seen-class validation macro-F1', 'tie_break': 'lower validation Benign FPR', 'test_used': False}, 'arms': {}}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Output folder:', OUTPUT_DIR)


## 6. Train one fresh five-output Task 1 model

Task 1 uses only old-class train/validation rows. Do not load an eight-output static checkpoint or pass future-class validation labels. The ordinary shuffled sampler remains default. Save the minimum-validation-loss checkpoint, verify class IDs 0–4, and compute Task 1 old-test scores only after selection. The same checkpoint starts both Task 2 arms.


In [ ]:
model_config = ModelConfig(num_features=len(tasks.feature_names), num_classes=5, hidden_dim=64, num_heads=4, num_layers=2, mlp_dim=128, batch_size=128, learning_rate=0.001, epochs=20, early_stopping_patience=5, device='cuda', mixed_precision=False, seed=42)
assert model_config.training_sampler == 'shuffled'
task1_model = TabularTransformerClassifier(model_config, class_ids=TASK1_CLASS_IDS)
torch.cuda.synchronize(); start = perf_counter()
task1_history = task1_model.fit(tasks.task1.train.features, tasks.task1.train.labels, tasks.task1.validation.features, tasks.task1.validation.labels, verbose=True)
torch.cuda.synchronize(); task1_seconds = perf_counter() - start
assert task1_model.class_ids == TASK1_CLASS_IDS
assert task1_history.best_epoch == int(np.argmin(task1_history.validation_loss)) + 1
assert np.isfinite(task1_history.train_loss + task1_history.validation_loss + task1_history.validation_accuracy).all()
task1_checkpoint = OUTPUT_DIR / 'task1_best_validation.pt'; task1_model.save(task1_checkpoint)
task1_predictions = task1_model.predict(tasks.task1.test.features)
task1_before = classification_metrics(tasks.task1.test.labels, task1_predictions, class_ids=TASK1_CLASS_IDS, class_names=tasks.class_names, benign_class_id=0)
task1_reloaded = TabularTransformerClassifier.load(task1_checkpoint, device='cuda')
assert task1_reloaded.class_ids == TASK1_CLASS_IDS and np.array_equal(task1_reloaded.predict(tasks.task1.test.features), task1_predictions)
(OUTPUT_DIR / 'task1_history.json').write_text(json.dumps(task1_history.to_dict(), indent=2), encoding='utf-8')
(OUTPUT_DIR / 'task1_metrics.json').write_text(json.dumps(task1_before, indent=2), encoding='utf-8')
manifest['task1'] = {'config': asdict(model_config), 'history': task1_history.to_dict(), 'old_test_metrics_before_task2': task1_before, 'training_seconds': task1_seconds, 'checkpoint': task1_checkpoint.name, 'reload_identical': True}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Task 1 best epoch:', task1_history.best_epoch, 'old-test accuracy:', task1_before['accuracy'])


## 7. Part B: calibrate and evaluate real held-out novelty **before expansion**

The Task 1 head still has only IDs 0–4. Calibrate the fifth percentile of *maximum class probability* from **Task 1 validation only**. Freeze it before evaluating known Task 1 test rows and genuinely unseen Task 2 test rows (IDs 5–7). The decision remains `max_probability < threshold`; equality counts as known. Neither future validation nor test rows fit preprocessing, train Task 1, or tune the threshold. Store binary unknown precision/recall/F1, known false-rejection rate, and per-unknown-class recall. We compute this now, before any Task 2 training/head expansion; results do not enter the separately predeclared strategy rule. Confidence thresholding may fail.


In [ ]:
assert task1_model.class_ids == TASK1_CLASS_IDS
validation_probabilities = task1_model.predict_proba(tasks.task1.validation.features)
novelty_calibration = calibrate_confidence_threshold(validation_probabilities, quantile=0.05)
known_probabilities = task1_model.predict_proba(tasks.task1.test.features)
unknown_probabilities = task1_model.predict_proba(tasks.task2.test.features)
assert known_probabilities.shape == (len(tasks.task1.test.labels), 5)
assert unknown_probabilities.shape == (len(tasks.task2.test.labels), 5)
assert np.array_equal(task1_predictions, np.asarray(task1_model.class_ids)[known_probabilities.argmax(axis=1)])
known_flags = detect_unknown(known_probabilities, novelty_calibration.threshold)
unknown_flags = detect_unknown(unknown_probabilities, novelty_calibration.threshold)
truth_flags = np.concatenate((np.zeros(len(known_flags), dtype=bool), np.ones(len(unknown_flags), dtype=bool)))
predicted_flags = np.concatenate((known_flags, unknown_flags))
truth_ids = np.concatenate((tasks.task1.test.labels, tasks.task2.test.labels))
novelty_result = novelty_metrics(truth_flags, predicted_flags, true_class_ids=truth_ids, unknown_class_ids=TASK2_CLASS_IDS)
novelty_record = {'calibration': asdict(novelty_calibration), 'known_class_ids': list(task1_model.class_ids), 'held_out_unknown_class_ids': list(TASK2_CLASS_IDS), 'metrics': novelty_result, 'protocol': 'known Task 1 test vs held-out Task 2 test, before head expansion'}
np.savez_compressed(OUTPUT_DIR / 'novelty_predictions.npz', known_original_rows=tasks.task1.test.row_indices, unknown_original_rows=tasks.task2.test.row_indices, known_max_confidence=known_probabilities.max(axis=1), unknown_max_confidence=unknown_probabilities.max(axis=1), known_flagged=known_flags, unknown_flagged=unknown_flags)
(OUTPUT_DIR / 'novelty.json').write_text(json.dumps(novelty_record, indent=2), encoding='utf-8')
manifest['held_out_novelty'] = novelty_record
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Frozen known-validation threshold:', novelty_calibration.threshold, 'known rows:', len(known_flags), 'held-out unknown rows:', len(unknown_flags))
print('Novelty metrics saved; they are not used for Task 2 arm selection')


## 8. Part A: train shuffled and class-balanced arms from identical Task 1 weights

Each arm independently reloads the same Task 1 checkpoint and expands to IDs 0–7 using the same seed. The complete expanded initial-weight hash must match. Both see identical clean Task 2 rows, the **same 500 unique replay examples**, clean seen-class validation, architecture, learning rate, batch, epochs, and patience. The shuffled arm uses existing concatenation/default behavior. Balanced mode weights every combined row by inverse frequency of its supplied class and uses seeded `WeightedRandomSampler(replacement=True, num_samples=4700)`, with no conflicting shuffle. A single generator advances across epochs. Draws can repeat examples; memory does not grow. Histories record expected class probabilities, actual counts each epoch, distinct replay rows ever drawn, total draws, and optimizer steps.


In [ ]:
def state_hash(model):
    digest = hashlib.sha256()
    for key, value in sorted(model.network.state_dict().items()):
        digest.update(key.encode()); digest.update(value.detach().cpu().numpy().tobytes())
    return digest.hexdigest()
arm_models = {}
arm_histories = {}
initial_hash = None
for name in ('shuffled', 'class_balanced'):
    folder = OUTPUT_DIR / name; folder.mkdir()
    model = TabularTransformerClassifier.load(task1_checkpoint, device='cuda')
    assert model.class_ids == TASK1_CLASS_IDS
    model.add_classes(TASK2_CLASS_IDS)
    assert model.class_ids == TASK1_CLASS_IDS + TASK2_CLASS_IDS
    expanded_hash = state_hash(model)
    if initial_hash is None:
        initial_hash = expanded_hash
    assert expanded_hash == initial_hash, 'Initial expanded weights differ across arms'
    if name == 'class_balanced':
        model.config = replace(model.config, training_sampler='class_balanced', sampler_seed=42)
    assert model.config.training_sampler == name
    torch.manual_seed(42); torch.cuda.manual_seed_all(42)
    torch.cuda.synchronize(); start = perf_counter()
    history = model.fit(tasks.task2.train.features, tasks.task2.train.labels, tasks.seen_validation.features, tasks.seen_validation.labels, replay=replay.as_fit_replay(), verbose=True)
    torch.cuda.synchronize(); train_seconds = perf_counter() - start
    assert model.class_ids == TASK1_CLASS_IDS + TASK2_CLASS_IDS
    assert history.best_epoch == int(np.argmin(history.validation_loss)) + 1
    assert np.isfinite(history.train_loss + history.validation_loss + history.validation_accuracy).all()
    assert history.replay_buffer_rows == 500 and history.training_draws == 4700 * history.epochs_completed
    assert history.optimizer_steps == 37 * history.epochs_completed
    assert sum(history.sampled_counts_by_class.values()) == history.training_draws
    assert len(history.sampled_counts_by_epoch) == history.epochs_completed
    if name == 'class_balanced':
        assert len(history.sampler_state_sha256_by_epoch) == history.epochs_completed
        assert len(set(history.sampler_state_sha256_by_epoch)) == history.epochs_completed, 'Sampler RNG did not advance across epochs'
        assert all(abs(value - 0.125) < 1e-10 for value in history.expected_class_probabilities.values())
    checkpoint = folder / 'best_validation.pt'; model.save(checkpoint)
    arm_models[name] = model; arm_histories[name] = history
    (folder / 'history.json').write_text(json.dumps(history.to_dict(), indent=2), encoding='utf-8')
    (folder / 'config.json').write_text(json.dumps({'model': asdict(model.config), 'source_sha': checked_out_sha, 'stored_replay_rows': 500}, indent=2), encoding='utf-8')
    manifest['arms'][name] = {'config': asdict(model.config), 'history': history.to_dict(), 'training_seconds': train_seconds, 'checkpoint': str(checkpoint.relative_to(OUTPUT_DIR)), 'expanded_initial_sha256': expanded_hash, 'stored_unique_replay_rows': 500, 'rows_per_epoch': 4700}
    (OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(name, 'epochs:', history.epochs_completed, 'draws:', history.training_draws, 'optimizer steps:', history.optimizer_steps, 'distinct replay rows drawn:', history.unique_replay_rows_drawn, 'counts:', history.sampled_counts_by_class)


## 9. Select the preferred strategy **only from validation**

The criterion was fixed before the run: higher clean seen-class validation macro-F1, then lower validation Benign FPR for an exact tie. Each model's own best checkpoint was already selected by clean validation loss inside `fit`. This step chooses between the two *strategies* using validation predictions only; no test metric or novelty result enters the rule. Both arms will still be reported on test afterward for transparency.


In [ ]:
ALL_IDS = TASK1_CLASS_IDS + TASK2_CLASS_IDS
validation_metrics = {}
for name, model in arm_models.items():
    predictions = model.predict(tasks.seen_validation.features)
    metrics = classification_metrics(tasks.seen_validation.labels, predictions, class_ids=ALL_IDS, class_names=tasks.class_names, benign_class_id=0)
    validation_metrics[name] = metrics
    (OUTPUT_DIR / name / 'validation_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
preferred_strategy = min(validation_metrics, key=lambda name: (-validation_metrics[name]['macro_f1'], validation_metrics[name]['benign_false_positive']['rate'], name))
selection = {'criterion': 'higher clean seen-class validation macro-F1; lower validation Benign FPR on exact tie', 'validation_metrics': validation_metrics, 'preferred_strategy': preferred_strategy, 'test_used': False, 'novelty_used': False}
(OUTPUT_DIR / 'strategy_selection.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')
manifest['strategy_selection'] = selection
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Validation-only preferred strategy:', preferred_strategy, 'macro-F1:', validation_metrics[preferred_strategy]['macro_f1'], 'Benign FPR:', validation_metrics[preferred_strategy]['benign_false_positive']['rate'])


## 10. Open Task 2 test results only after strategy selection

Report old/new/combined accuracy and macro-F1, per-class recall/F1, Benign FPR, and signed forgetting for both arms. Old-class macro-F1 averages the same five classes before/after; new-class predictions on old test rows remain errors. Verify each checkpoint reload on combined test predictions. Test scores describe this fixed comparison; they do not change the validation-only preferred strategy.


In [ ]:
def checked_predictions(model, part):
    probabilities = model.predict_proba(part.features)
    predictions = model.predict(part.features)
    assert probabilities.shape == (len(part.labels), len(model.class_ids))
    assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5)
    assert np.array_equal(predictions, np.asarray(model.class_ids)[probabilities.argmax(axis=1)])
    return predictions
def focused_metrics(truth, predictions, focus_ids, *, benign=False):
    result = classification_metrics(truth, predictions, class_ids=ALL_IDS, class_names=tasks.class_names, benign_class_id=0 if benign else None)
    result['focused_class_ids'] = list(focus_ids)
    result['focused_macro_f1'] = float(np.mean([result['per_class'][str(class_id)]['f1'] for class_id in focus_ids]))
    return result
test_summary = {}
for name, model in arm_models.items():
    folder = OUTPUT_DIR / name
    old_pred = checked_predictions(model, tasks.task1.test)
    new_pred = checked_predictions(model, tasks.task2.test)
    combined_pred = checked_predictions(model, tasks.combined_test)
    old_metrics = focused_metrics(tasks.task1.test.labels, old_pred, TASK1_CLASS_IDS, benign=True)
    new_metrics = focused_metrics(tasks.task2.test.labels, new_pred, TASK2_CLASS_IDS)
    combined_metrics = classification_metrics(tasks.combined_test.labels, combined_pred, class_ids=ALL_IDS, class_names=tasks.class_names, benign_class_id=0)
    forgetting = forgetting_metrics(tasks.task1.test.labels, task1_predictions, old_pred, old_class_ids=TASK1_CLASS_IDS, all_class_ids=ALL_IDS)
    reloaded = TabularTransformerClassifier.load(folder / 'best_validation.pt', device='cuda')
    reload_identical = reloaded.class_ids == model.class_ids and np.array_equal(reloaded.predict(tasks.combined_test.features), combined_pred)
    assert reload_identical, f'{name}: checkpoint reload mismatch'
    metrics = {'task1_before': task1_before, 'task1_after': old_metrics, 'task2_after': new_metrics, 'combined_after': combined_metrics, 'forgetting': forgetting}
    (folder / 'test_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    manifest['arms'][name]['test_metrics'] = metrics
    manifest['arms'][name]['checkpoint_reload_identical'] = bool(reload_identical)
    (OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    test_summary[name] = {'old_accuracy': old_metrics['accuracy'], 'old_macro_f1': old_metrics['focused_macro_f1'], 'new_accuracy': new_metrics['accuracy'], 'new_macro_f1': new_metrics['focused_macro_f1'], 'combined_accuracy': combined_metrics['accuracy'], 'combined_macro_f1': combined_metrics['macro_f1'], 'benign_fpr': old_metrics['benign_false_positive']['rate'], 'accuracy_forgetting': forgetting['accuracy_forgetting'], 'macro_f1_forgetting': forgetting['macro_f1_forgetting'], 'training_draws': arm_histories[name].training_draws, 'optimizer_steps': arm_histories[name].optimizer_steps, 'unique_replay_rows_drawn': arm_histories[name].unique_replay_rows_drawn}
    print(name, test_summary[name])
(OUTPUT_DIR / 'test_summary.json').write_text(json.dumps(test_summary, indent=2), encoding='utf-8')
display(pd.DataFrame(test_summary).T)
print('Validation-only preferred strategy remains:', preferred_strategy)


## 11. Plot observed exposure and novelty confidence; collect the run

Plots make the actual sampling change visible. Expected probabilities are targets, not guarantees: weighted draws use replacement and can repeat rows, while the 500-row replay memory remains fixed. The novelty histogram uses only the frozen Task 1 model and threshold. Send the executed notebook and full timestamped output folder back before streaming preparation; do not infer unknown-detection quality until these real held-out metrics are inspected.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for name, history in arm_histories.items():
    epochs = np.arange(1, history.epochs_completed + 1)
    axes[0].plot(epochs, history.validation_loss, marker='o', label=name)
axes[0].set(title='Clean seen-class validation loss', xlabel='Epoch', ylabel='Cross-entropy')
axes[0].legend()
for name, history in arm_histories.items():
    total = history.training_draws
    axes[1].plot(list(ALL_IDS), [history.sampled_counts_by_class[class_id] / total for class_id in ALL_IDS], marker='o', label=name)
axes[1].set(title='Actual training draw share', xlabel='Global class ID', ylabel='Draw fraction', ylim=(0, 0.4))
axes[1].legend()
axes[2].hist(known_probabilities.max(axis=1), bins=30, alpha=0.6, label='Known Task 1 test')
axes[2].hist(unknown_probabilities.max(axis=1), bins=30, alpha=0.6, label='Held-out Task 2 test')
axes[2].axvline(novelty_calibration.threshold, color='black', linestyle='--', label='Frozen 5th percentile')
axes[2].set(title='Task 1 maximum confidence', xlabel='Maximum probability', ylabel='Rows')
axes[2].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'training_exposure_novelty.png', dpi=160); plt.show()
manifest['artifacts'] = ['manifest.json', 'row_indices.npz', 'replay_buffer.npz', 'preprocessing.json', 'task1_best_validation.pt', 'task1_history.json', 'task1_metrics.json', 'novelty.json', 'novelty_predictions.npz', 'strategy_selection.json', 'test_summary.json', 'shuffled/history.json', 'shuffled/validation_metrics.json', 'shuffled/test_metrics.json', 'shuffled/best_validation.pt', 'class_balanced/history.json', 'class_balanced/validation_metrics.json', 'class_balanced/test_metrics.json', 'class_balanced/best_validation.pt', 'training_exposure_novelty.png']
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Source SHA:', checked_out_sha, 'GPU:', environment['gpu_name'], 'input SHA-256:', input_sha256)
print('Validation-only preferred strategy:', preferred_strategy)
print('Held-out unknown precision/recall/F1:', novelty_result['unknown_precision'], novelty_result['unknown_recall'], novelty_result['unknown_f1'])
print('Known false-rejection rate:', novelty_result['known_false_rejection_rate'], 'per unknown class:', novelty_result['per_unknown_class'])
print('Task 1 and both Task 2 checkpoint reloads: identical; output:', OUTPUT_DIR)
